Import libraries and functions

In [1]:
import numpy as np
import torch
import pyro
from torch import nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from codebase_0 import (
    CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
)
from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, to_device
)
from gamma_bcann_vi import GammaBCANN, gamma_build_bcann_guide, gamma_fit_glm_mle
from bcann_neutra import neutra_fit_guide, time_neutra, run_neutra_parallel


seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load dataset and train test split

In [2]:
device = torch.device("cuda:3")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 10000
)

Create GammaBCANN and fit a IAF variational posterior

In [4]:
network_architecture = [20, 15, 10]

# flow (step 2)
NUM_TRANSFORMS = 2        # IAF layers; 0 -> diagonal Gaussian (NeuTra only rescales the space)
FLOW_HIDDEN = None       # IAF hidden widths; None -> [latent_dim]. Try [512] if the flow is slow
SVI_BATCH = None          # None -> full-batch ELBO on the subsample
SVI_STEPS = 5000
SVI_LR, SVI_LR_FINAL = 5e-3, 5e-4

glm_mle = gamma_fit_glm_mle(X_learn, y_learn, w_learn, random_state=seed)

with torch.device(device):
        gamma_bcann = GammaBCANN(
                in_features= 48,
                neural_net_params={"hidden_activation_nodes": network_architecture,
                "hidden_activations": nn.Tanh,
                "prior_scale": 1.0},
        )
        gamma_guide, gamma_history = neutra_fit_guide(
                gamma_bcann, 
                X_learn, 
                y_learn, 
                w_learn, 
                glm_mle=tuple(t.to(device) for t in glm_mle),
                num_transforms=NUM_TRANSFORMS, hidden_dim= FLOW_HIDDEN,
                num_steps=SVI_STEPS, lr=SVI_LR, lr_final=SVI_LR_FINAL, batch_size=10000,
                report_every=max(SVI_STEPS // 20, 1), random_state=seed
        )


step 250 | -ELBO/n: 8.6067 | 8s
step 500 | -ELBO/n: 8.6023 | 16s
step 750 | -ELBO/n: 8.5977 | 24s
step 1000 | -ELBO/n: 8.6023 | 32s
step 1250 | -ELBO/n: 8.5994 | 40s
step 1500 | -ELBO/n: 8.5999 | 48s
step 1750 | -ELBO/n: 8.6004 | 56s
step 2000 | -ELBO/n: 8.6009 | 64s
step 2250 | -ELBO/n: 8.5982 | 71s
step 2500 | -ELBO/n: 8.6003 | 80s
step 2750 | -ELBO/n: 8.5967 | 88s
step 3000 | -ELBO/n: 8.5964 | 95s
step 3250 | -ELBO/n: 8.5984 | 103s
step 3500 | -ELBO/n: 8.5986 | 111s
step 3750 | -ELBO/n: 8.5996 | 119s
step 4000 | -ELBO/n: 8.5977 | 126s
step 4250 | -ELBO/n: 8.5978 | 134s
step 4500 | -ELBO/n: 8.6005 | 142s
step 4750 | -ELBO/n: 8.6029 | 150s
step 5000 | -ELBO/n: 8.5974 | 158s


Create NeuTra hyperparameters

In [6]:
# NUTS (step 3)
CHAINS = 6
WARMUP, SAMPLES = 1000, 1000
TARGET_ACCEPT = 0.8
MAX_TREE_DEPTH = 8

nuts_kwargs = dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, init_scale=1.0, jit_compile = True)


Run NeuTra on the learning set

In [7]:
neutra_gamma_samples, neutra_gamma_info = run_neutra_parallel(
    GammaBCANN, 
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh,
        }
    ), 
    gamma_guide, 
    X_learn, 
    y_learn,
    w_learn,
    num_chains=CHAINS, 
    devices=device,
    out_dir="gamma_neutra_chains", 
    seed=seed,
    **nuts_kwargs
)

elapsed 0:52:25
chain 5: Sample: 100%|██████████| 2000/2000 [52:02,  1.56s/it, step size=3.58e-02, acc. prob=0.944]


Save to disk

In [8]:
torch.save(
    {
        "samples": neutra_gamma_samples,
        "info": neutra_gamma_info,
        "config": dict(num_warmup=1000, num_samples=1000, target_accept=0.8,
                       max_tree_depth=8, seed=seed, n_chains=6),
    },
    "hmc_results/gamma_neutra_full_set.pt",
)
